# DBSCAN Experiments (Thesis-aligned)

Notebook ini fokus pada alur: 1) impor dan konfigurasi, 2) k-distance plot untuk menentukan eps, 3) eksperimen parameter (eps & min_samples) menggunakan silhouette (sample), 4) fit final DBSCAN dan simpan model, 5) evaluasi cluster.

In [ ]:
# Part 1 — Imports & configuration
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd
import gc
from tqdm import tqdm
import time

# ============================================================================
# CONFIGURATION - Specify your embedding file(s) directly
# ============================================================================

# Option 1: Single file (BGL or Thunderbird)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
    # Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_thunderbird_embeddings.npy"),
]

# Option 2: Multiple files (Combined dataset)
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_thunderbird_embeddings.npy"),
# ]

# Option 3: PCA variants (smaller, faster - RECOMMENDED for DBSCAN!)
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/after_preprocessed_bgl_pca256_embeddings.npy"),
# ]

# Option 4: PCA128 for ultra-large datasets
# INPUT_FILES = [
#     Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca128/after_preprocessed_thunderbird_pca128_embeddings.npy"),
# ]

RANDOM_STATE = 42
SAMPLE_FOR_METRICS = 50000  # DBSCAN: reduce if memory limited
SAMPLE_FOR_KDIST = 100000   # Sample for k-distance plot (large datasets)
KNN_NEIGHBORS = 4  # for k-distance plot (k = min_samples)

print("📁 Input files configured:")
for f in INPUT_FILES:
    if f.exists():
        size_gb = f.stat().st_size / (1024**3)
        print(f"  ✓ {f.name} ({size_gb:.2f} GB)")
    else:

        print(f"  ❌ NOT FOUND: {f}")

## Test: File Detection & Size Analysis

Quick diagnostic untuk verify files dan estimate runtime.

In [ ]:
# Diagnostic: Analyze configured files with auto-dimension detection
print("🔍 Analyzing configured files for DBSCAN...\n")

def detect_embedding_dim_diag(file_path: Path) -> int:
    """Auto-detect embedding dimension from filename pattern"""
    filename = file_path.name.lower()
    if 'pca256' in filename:
        return 256
    elif 'pca128' in filename:
        return 128
    else:
        return 768

def infer_num_rows_diag(path: Path, embedding_dim: int = None) -> int:
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim_diag(path)
    size = path.stat().st_size
    return size // (embedding_dim * np.dtype(np.float32).itemsize)

total_size_gb = 0
file_details = []

for f in INPUT_FILES:
    if not f.exists():
        print(f"❌ File not found: {f}")
        continue
        
    size_gb = f.stat().st_size / (1024**3)
    total_size_gb += size_gb
    
    # Auto-detect dimension from filename
    auto_dim = detect_embedding_dim_diag(f)
    
    try:
        test_arr = np.load(f, mmap_mode='r')
        file_type = "Standard .npy"
        shape = test_arr.shape
        actual_dim = test_arr.shape[1]
        del test_arr
        
        # Verify auto-detection matches
        if actual_dim != auto_dim:
            print(f"⚠️ Dimension mismatch for {f.name}:")
            print(f"   Auto-detected: {auto_dim}, Actual: {actual_dim}")
            print(f"   Using actual dimension from .npy header")
    except:
        file_type = "RAW memmap"
        n_rows = infer_num_rows_diag(f, embedding_dim=auto_dim)
        shape = (n_rows, auto_dim)
        actual_dim = auto_dim
    
    file_details.append({
        'name': f.name,
        'size_gb': size_gb,
        'type': file_type,
        'shape': shape,
        'dimension': actual_dim
    })

if len(file_details) == 0:
    print("⚠️ No valid files found! Check INPUT_FILES configuration.")
else:
    df = pd.DataFrame(file_details)
    print(df.to_string(index=False))
    
    print(f"\n{'='*60}")
    print(f"Total files: {len(file_details)}")
    print(f"Total size: {total_size_gb:.2f} GB")
    print(f"Total samples: {sum(d['shape'][0] for d in file_details):,}")
    
    # DBSCAN-specific warnings
    print(f"\n📊 DBSCAN RUNTIME ESTIMATE:")
    if total_size_gb < 5:
        print("✅ Small dataset (<5GB)")
        print("   → Expected time: 10-20 minutes")
        print("   → Memory: ~{:.1f} GB RAM".format(total_size_gb * 2))
    elif total_size_gb < 20:
        print("⚠️  Medium dataset (5-20GB)")
        print("   → Expected time: 30-60 minutes")
        print("   → Memory: ~{:.1f} GB RAM".format(total_size_gb * 2))
    elif total_size_gb < 100:
        print("🔥 Large dataset (20-100GB)")
        print("   → Expected time: 1-3 hours")
        print("   → Memory: ~{:.1f} GB RAM".format(total_size_gb * 2))
        print("   ⚠️ Consider using PCA variants!")
    else:
        print("❌ ULTRA LARGE dataset (>100GB)")
        print("   → Expected time: 4-8+ hours")
        print("   → Memory: ~{:.1f} GB RAM".format(total_size_gb * 2))
        print("   ⚠️ STRONGLY RECOMMEND: Use PCA128 variant!")
    
    print(f"\n💡 RECOMMENDATION:")
    if total_size_gb > 20:
        print("   Switch to PCA variant for faster processing:")
        print("   • PCA256: ~6x smaller, minimal quality loss")
        print("   • PCA128: ~12x smaller, acceptable quality loss")


In [ ]:
# Part 2 — Smart file loading with auto-dimension detection and k-distance plot to estimate eps

def detect_embedding_dim(file_path: Path) -> int:
    """
    Auto-detect embedding dimension from filename pattern
    - *pca256* → 256 dims
    - *pca128* → 128 dims
    - default → 768 dims
    """
    filename = file_path.name.lower()
    if 'pca256' in filename:
        return 256
    elif 'pca128' in filename:
        return 128
    else:
        return 768

def infer_num_rows(path: Path, embedding_dim: int = None) -> int:
    """
    Infer number of rows for RAW memmap files
    Auto-detects dimension from filename if not provided
    """
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(path)
    size = path.stat().st_size
    return size // (embedding_dim * np.dtype(np.float32).itemsize)

def load_single_file_smart(file_path: Path, embedding_dim: int = None):
    """
    Smart loader: auto-detect .npy vs RAW memmap
    Returns (array, is_memmap, num_rows)
    
    Auto-detects dimension from filename if not provided:
    - *pca256* → 256 dims
    - *pca128* → 128 dims  
    - default → 768 dims
    """
    # Auto-detect dimension if not provided
    if embedding_dim is None:
        embedding_dim = detect_embedding_dim(file_path)
        print(f"   🔍 Auto-detected dimension: {embedding_dim} from filename")
    
    try:
        arr = np.load(file_path, mmap_mode='r')
        detected_dim = arr.shape[1]
        if detected_dim != embedding_dim:
            print(f"   ⚠️ Dimension mismatch! Expected {embedding_dim}, got {detected_dim} from .npy header")
            embedding_dim = detected_dim
        return arr, True, arr.shape[0]
    except Exception:
        num_rows = infer_num_rows(file_path, embedding_dim)
        arr = np.memmap(
            file_path, 
            dtype=np.float32, 
            mode='r', 
            shape=(num_rows, embedding_dim)
        )
        print(f"   ⚠️ Loaded as RAW memmap: {num_rows:,} rows × {embedding_dim} dims")
        return arr, True, num_rows

def load_embeddings_from_files(files, force_copy=False):
    """Load embeddings from list of file paths with auto-dimension detection"""
    if len(files) == 0:
        raise FileNotFoundError('No embedding files provided')
    
    for f in files:
        if not f.exists():
            raise FileNotFoundError(f'File not found: {f}')
    
    # Auto-detect dimension from first file
    first_arr, _, _ = load_single_file_smart(files[0])
    embedding_dim = first_arr.shape[1]
    print(f"Embedding dimension: {embedding_dim}")
    
    if len(files) == 1:
        print(f"Loading single file: {files[0].name}")
        return first_arr
    
    print(f"Loading {len(files)} files...")
    total_samples = 0
    file_info = []
    for f in files:
        arr, is_mmap, n_rows = load_single_file_smart(f)  # Auto-detect dimension
        file_info.append((f, arr, n_rows))
        total_samples += n_rows
        print(f"  - {f.name}: {n_rows:,} rows")
    
    total_size_gb = (total_samples * embedding_dim * 4) / (1024**3)
    print(f"\nTotal samples: {total_samples:,} ({total_size_gb:.2f} GB)")
    
    if total_size_gb < 100:
        print("Strategy: Memory-mapped stacking")
        arrays = [arr for _, arr, _ in file_info]
        return np.vstack(arrays)
    else:
        raise MemoryError(
            f"Dataset too large ({total_size_gb:.1f}GB). "
            "For DBSCAN, use single file or PCA variants (smaller size)"
        )

# Load embeddings
print("Loading embeddings...")
emb = load_embeddings_from_files(INPUT_FILES)
print(f'Loaded embeddings shape: {emb.shape}')

# For very large datasets, sample for k-distance plot
n_total = emb.shape[0]
if n_total > SAMPLE_FOR_KDIST:
    print(f'\n⚠️ Large dataset ({n_total:,} samples)')
    print(f'   Using sample of {SAMPLE_FOR_KDIST:,} for k-distance plot')
    rng = np.random.RandomState(RANDOM_STATE)
    kdist_idx = rng.choice(n_total, SAMPLE_FOR_KDIST, replace=False)
    emb_for_kdist = emb[kdist_idx]
else:
    emb_for_kdist = emb

# Compute nearest-neighbors distances (k-distance)
print(f'\nComputing k-distance (k={KNN_NEIGHBORS})...')
nn = NearestNeighbors(n_neighbors=KNN_NEIGHBORS, n_jobs=-1)
nn.fit(emb_for_kdist)
distances, _ = nn.kneighbors(emb_for_kdist)
# distances[:, -1] is the distance to k-th neighbor
k_dist = np.sort(distances[:, -1])

# Plot k-distance curve
plt.figure(figsize=(10,4))
plt.plot(k_dist)
plt.xlabel('Points sorted by k-distance')
plt.ylabel(f'k-distance (k={KNN_NEIGHBORS})')
plt.title('k-distance plot — look for elbow to choose eps')
plt.axhline(y=np.percentile(k_dist, 90), color='r', linestyle='--', alpha=0.5, label='90th percentile')
plt.axhline(y=np.percentile(k_dist, 95), color='orange', linestyle='--', alpha=0.5, label='95th percentile')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n📊 k-distance statistics:')
print(f'   50th percentile: {np.percentile(k_dist, 50):.4f}')
print(f'   75th percentile: {np.percentile(k_dist, 75):.4f}')
print(f'   90th percentile: {np.percentile(k_dist, 90):.4f}')
print(f'   95th percentile: {np.percentile(k_dist, 95):.4f}')


In [ ]:
# Part 3 — Parameter search over eps and min_samples (uses a sample for silhouette)
# WARNING: DBSCAN can be slow for large datasets; we sample for metric computation

print('Preparing sample for parameter search...')
n = emb.shape[0]
if n > SAMPLE_FOR_METRICS:
    rng = np.random.RandomState(RANDOM_STATE)
    sample_idx = rng.choice(n, SAMPLE_FOR_METRICS, replace=False)
    emb_sample = emb[sample_idx]
    print(f'Using sample: {SAMPLE_FOR_METRICS:,} / {n:,} samples')
else:
    emb_sample = emb
    print(f'Using full dataset: {n:,} samples')

# Define parameter grid based on k-distance statistics
eps_min = np.percentile(k_dist, 50)
eps_max = np.percentile(k_dist, 95)
eps_values = np.linspace(eps_min, eps_max, 8)
min_samples_values = [3, 5, 8, 12]

print(f'\nParameter grid:')
print(f'  eps: {len(eps_values)} values from {eps_min:.4f} to {eps_max:.4f}')
print(f'  min_samples: {min_samples_values}')
print(f'  Total combinations: {len(eps_values) * len(min_samples_values)}')

results = []

total_combinations = len(eps_values) * len(min_samples_values)
print(f'\n🔄 Testing {total_combinations} parameter combinations...')

pbar = tqdm(total=total_combinations, desc='DBSCAN grid search', unit='config')
for eps_idx, eps in enumerate(eps_values, 1):
    for ms in min_samples_values:
        start_time = time.time()
        
        # n_jobs=-1 to use all CPU cores
        dbs = DBSCAN(eps=float(eps), min_samples=int(ms), n_jobs=-1)
        labels = dbs.fit_predict(emb_sample)
        
        # Compute number of clusters (exclude noise label -1)
        unique_labels = set(labels) - {-1}
        n_clusters = len(unique_labels)
        n_noise = np.sum(labels == -1)
        noise_pct = (n_noise / len(labels)) * 100
        
        sil = -1
        if n_clusters > 1:
            try:
                # n_jobs=-1 to maximize CPU usage
                sil = silhouette_score(emb_sample, labels, n_jobs=-1)
            except Exception:
                sil = -1
        
        results.append({
            'eps': float(eps),
            'min_samples': int(ms),
            'n_clusters': n_clusters,
            'noise_pct': float(noise_pct),
            'silhouette': float(sil)
        })
        
        elapsed = time.time() - start_time
        pbar.update(1)
        tqdm.write(f'  eps={eps:.4g}, min_samples={ms:2d} → clusters={n_clusters:2d}, noise={noise_pct:5.1f}%, sil={sil:6.4f} ({elapsed:.1f}s)')

pbar.close()

# Show results sorted by silhouette
df_res = pd.DataFrame(results)
print('\n' + '='*70)
print('TOP 10 CONFIGURATIONS (by silhouette score)')
print('='*70)
print(df_res.sort_values('silhouette', ascending=False).head(10).to_string(index=False))

# Also show configurations with reasonable noise levels
print('\n' + '='*70)
print('CONFIGURATIONS WITH LOW NOISE (<30%)')
print('='*70)
df_low_noise = df_res[df_res['noise_pct'] < 30].sort_values('silhouette', ascending=False)
if len(df_low_noise) > 0:
    print(df_low_noise.head(10).to_string(index=False))
else:
    print('⚠️ No configurations with <30% noise found')

In [ ]:
# Part 4 — Fit final DBSCAN on full data and save model+labels

# Set chosen parameters based on Part 3 results
# Option 1: Best silhouette score
best_config = df_res.sort_values('silhouette', ascending=False).iloc[0]

# Option 2: Best with low noise (uncomment if preferred)
# df_low_noise = df_res[df_res['noise_pct'] < 30].sort_values('silhouette', ascending=False)
# if len(df_low_noise) > 0:
#     best_config = df_low_noise.iloc[0]
# else:
#     print('⚠️ No low-noise config, using best silhouette')
#     best_config = df_res.sort_values('silhouette', ascending=False).iloc[0]

CHOSEN_EPS = float(best_config['eps'])
CHOSEN_MIN_SAMPLES = int(best_config['min_samples'])

print('='*70)
print('FINAL DBSCAN CONFIGURATION')
print('='*70)
print(f'eps:         {CHOSEN_EPS:.6f}')
print(f'min_samples: {CHOSEN_MIN_SAMPLES}')
print(f'Expected clusters: {int(best_config["n_clusters"])}')
print(f'Expected noise:    {best_config["noise_pct"]:.1f}%')
print(f'Expected silhouette: {best_config["silhouette"]:.4f}')
print('='*70)

print(f'\n🔄 Fitting DBSCAN on full dataset ({emb.shape[0]:,} samples)...')
print('⏳ This may take a while... Monitor CPU usage in htop/Task Manager')
start_time = time.time()
# n_jobs=-1 to use all CPU cores
model = DBSCAN(eps=CHOSEN_EPS, min_samples=CHOSEN_MIN_SAMPLES, n_jobs=-1)
labels_full = model.fit_predict(emb)
elapsed = time.time() - start_time
print(f'✓ DBSCAN completed in {elapsed/60:.1f} minutes')

# Save labels and model
out_model = Path('dbscan_model.pkl')
joblib.dump(model, out_model)
np.save('dbscan_labels.npy', labels_full)
np.save('dbscan_config.npy', np.array([CHOSEN_EPS, CHOSEN_MIN_SAMPLES]))

print(f'\n✓ Saved: {out_model}')
print(f'✓ Saved: dbscan_labels.npy ({len(labels_full):,} labels)')
print(f'✓ Saved: dbscan_config.npy (eps, min_samples)')

In [ ]:
# Part 5 — Evaluation and cluster analysis

from collections import Counter
counts = Counter(labels_full)

print('='*70)
print('CLUSTER ANALYSIS')
print('='*70)
print('\nLabel counts (including noise -1):')
for lbl, cnt in sorted(counts.items()):
    pct = (cnt / len(labels_full)) * 100
    cluster_type = 'NOISE' if lbl == -1 else f'Cluster {lbl}'
    print(f' - {cluster_type:12s}: {cnt:8,} samples ({pct:5.2f}%)')

# Separate noise and clusters
n_noise = counts.get(-1, 0)
n_clusters = len(set(labels_full) - {-1})
n_clustered = len(labels_full) - n_noise

print(f'\nSummary:')
print(f'  Total samples:    {len(labels_full):,}')
print(f'  Clusters found:   {n_clusters}')
print(f'  Clustered points: {n_clustered:,} ({(n_clustered/len(labels_full)*100):.1f}%)')
print(f'  Noise points:     {n_noise:,} ({(n_noise/len(labels_full)*100):.1f}%)')

# Compute metrics on sample or full dataset
n = emb.shape[0]
if n > SAMPLE_FOR_METRICS:
    print(f'\n📊 Computing metrics on sample ({SAMPLE_FOR_METRICS:,} samples)...')
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.choice(n, SAMPLE_FOR_METRICS, replace=False)
    lbls_s = labels_full[idx]
    emb_s = emb[idx]
else:
    print(f'\n📊 Computing metrics on full dataset...')
    emb_s = emb
    lbls_s = labels_full

if len(set(lbls_s) - {-1}) > 1:
    try:
        sil = silhouette_score(emb_s, lbls_s)
        print(f'  Silhouette Score: {sil:.4f}')
    except Exception as e:
        print(f'  Silhouette Score: N/A ({e})')
    
    try:
        dbi = davies_bouldin_score(emb_s, lbls_s)
        print(f'  Davies-Bouldin Index: {dbi:.4f} (lower is better)')
    except Exception as e:
        print(f'  Davies-Bouldin Index: N/A ({e})')
else:
    print('⚠️ Not enough clusters (excluding noise) to compute metrics')

# Visualize cluster size distribution (excluding noise)
cluster_labels = [lbl for lbl in labels_full if lbl != -1]
if len(cluster_labels) > 0:
    cluster_counts = Counter(cluster_labels)
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.bar(cluster_counts.keys(), cluster_counts.values())
    plt.xlabel('Cluster ID')
    plt.ylabel('Number of samples')
    plt.title('Cluster Size Distribution')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    sizes = list(cluster_counts.values())
    plt.hist(sizes, bins=min(20, len(sizes)), edgecolor='black')
    plt.xlabel('Cluster size')
    plt.ylabel('Frequency')
    plt.title('Cluster Size Histogram')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'\nCluster size statistics:')
    print(f'  Min:    {min(sizes):,}')
    print(f'  Max:    {max(sizes):,}')
    print(f'  Mean:   {np.mean(sizes):,.1f}')
    print(f'  Median: {np.median(sizes):,.1f}')


## Configuration Examples & Recommended Settings

### **Quick Start: How to Use This Notebook**

Edit **Cell 2** dan ganti `INPUT_FILES` sesuai kebutuhan:

```python
# Example 1: BGL only (12GB - fast)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector/after_preprocessed_bgl_embeddings.npy"),
]

# Example 2: BGL PCA256 (4GB - RECOMMENDED for DBSCAN!)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca256/after_preprocessed_bgl_pca256_embeddings.npy"),
]

# Example 3: Thunderbird PCA128 (100GB - possible but slow)
INPUT_FILES = [
    Path("/media/bioinfo04/Expansion/2427051003_dataset_vector_pca128/after_preprocessed_thunderbird_pca128_embeddings.npy"),
]
```

### **⚠️ DBSCAN Important Notes:**

1. **DBSCAN is MEMORY INTENSIVE** - bekerja pada full dataset, tidak bisa incremental
2. **Recommendation:** Gunakan **PCA variants** untuk speed:
   - BGL baseline (12GB) → ~30-45 min
   - BGL PCA256 (4GB) → ~10-15 min ✅ BEST
   - Thunderbird baseline (600GB) → ❌ TOO SLOW (hours!)
   - Thunderbird PCA128 (100GB) → ~2-3 hours (possible)

3. **Sampling Strategy:**
   - k-distance plot: Sample 100K untuk speed
   - Parameter search: Sample 50K (adjustable)
   - Final fit: Full dataset

### **Recommended Configurations:**

#### **1. BGL PCA256 (Fast & Quality)**
```python
INPUT_FILES = [
    Path("/media/.../dataset_vector_pca256/after_preprocessed_bgl_pca256_embeddings.npy"),
]
SAMPLE_FOR_METRICS = 50000
SAMPLE_FOR_KDIST = 100000
```
**Time:** 10-15 minutes | **Quality:** ⭐⭐⭐⭐⭐

---

#### **2. BGL Baseline (High Quality, Slower)**
```python
INPUT_FILES = [
    Path("/media/.../dataset_vector/after_preprocessed_bgl_embeddings.npy"),
]
SAMPLE_FOR_METRICS = 50000
SAMPLE_FOR_KDIST = 100000
```
**Time:** 30-45 minutes | **Quality:** ⭐⭐⭐⭐⭐

---

#### **3. Thunderbird PCA128 (Large Dataset)**
```python
INPUT_FILES = [
    Path("/media/.../dataset_vector_pca128/after_preprocessed_thunderbird_pca128_embeddings.npy"),
]
SAMPLE_FOR_METRICS = 50000   # Keep small
SAMPLE_FOR_KDIST = 200000     # Can increase
```
**Time:** 2-3 hours | **Quality:** ⭐⭐⭐⭐

---

### **Parameter Tuning Guidelines:**

**eps (epsilon):**
- Look at k-distance plot elbow
- Typical range: 50th to 95th percentile of k-distances
- Smaller eps → more clusters, more noise
- Larger eps → fewer clusters, less noise

**min_samples:**
- Recommended: 4-12 for log data
- Smaller values → more sensitive, more clusters
- Larger values → more robust to noise

**Trade-offs:**
- High silhouette but high noise → overfitting
- Low noise but low clusters → too loose
- **Balance:** Aim for <30% noise with good silhouette

---

### **Workflow:**
```
1. Run Cell 2: Load data & k-distance plot
   └─> Observe elbow point
   
2. Run Cell 3: Parameter search
   └─> Review top configurations
   └─> Consider both silhouette AND noise %
   
3. Run Cell 4: Fit final model
   └─> Auto-selects best config
   └─> Can manually override if needed
   
4. Run Cell 5: Evaluate results
   └─> Check cluster sizes
   └─> Identify anomaly clusters (small clusters)
```

---

### **Output Files:**
- `dbscan_model.pkl` - Trained DBSCAN model
- `dbscan_labels.npy` - Cluster labels (including -1 for noise)
- `dbscan_config.npy` - Final eps & min_samples values

### **Anomaly Detection Strategy:**
1. **Noise points (label -1)** → Likely anomalies
2. **Small clusters** → Potential anomaly patterns
3. **Large clusters** → Normal behavior patterns

Untuk inference:
```python
model = joblib.load('dbscan_model.pkl')
new_labels = model.fit_predict(new_embeddings)  # Note: DBSCAN requires fit_predict
```